In [ ]:
# Example 1: Basic Flask application with metrics
from prometheus_flask_exporter import PrometheusMetrics
from flask import Flask, jsonify, request
import time
import requests
import threading
from werkzeug.serving import make_server

# Create Flask application
app = Flask(__name__)

# Initialize Prometheus metrics
metrics = PrometheusMetrics(app)

# Define routes with automatic metrics
@app.route('/')
def home():
    return jsonify({'message': 'Hello from Flask with Prometheus metrics!'})

@app.route('/users/<user_id>')
def get_user(user_id):
    # Simulate variable response time
    time.sleep(0.1)
    return jsonify({'user_id': user_id, 'name': f'User {user_id}'})

@app.route('/slow')
def slow_endpoint():
    time.sleep(0.5)
    return jsonify({'message': 'This is a slow endpoint'})

print("Basic Flask application with automatic Prometheus metrics")
print("=" * 55)

server = make_server('127.0.0.1', 0, app, threaded=True)
_port = server.socket.getsockname()[1]
server_thread = threading.Thread(target=server.serve_forever)
server_thread.daemon = True
server_thread.start()

time.sleep(0.5)  # Wait for server to start

# Make test requests to generate metrics
test_urls = [
    f'http://127.0.0.1:{_port}/',
    f'http://127.0.0.1:{_port}/users/123',
    f'http://127.0.0.1:{_port}/users/456',
    f'http://127.0.0.1:{_port}/slow'
]

print("Making test requests to generate metrics:")
for url in test_urls:
    response = requests.get(url, timeout=2)
    print(f"  {url} -> {response.status_code}")
    assert response.status_code == 200, (url, response.status_code)
assert requests.get(test_urls[1], timeout=2).json() == {'user_id': '123', 'name': 'User 123'}

# Check metrics endpoint
metrics_response = requests.get(f'http://127.0.0.1:{_port}/metrics', timeout=2)
metrics_data = metrics_response.text

print(f"\nMetrics endpoint status: {metrics_response.status_code}")
assert metrics_response.status_code == 200, metrics_response.status_code
assert metrics_response.headers['Content-Type'].startswith('text/plain')
print("Sample metrics (first 500 chars):")
print(metrics_data[:500] + "..." if len(metrics_data) > 500 else metrics_data)

# Count metric lines
metric_lines = [line for line in metrics_data.split('\n') if line and not line.startswith('#')]
print(f"Total metric lines: {len(metric_lines)}")

# Five requests so far (the JSON check above requested /users/123 a second time);
# the /metrics endpoint itself is not counted
print("\nKey metrics captured:")
assert 'flask_http_request_total{method="GET",status="200"} 5.0' in metrics_data
for path, count in [('/', 1), ('/users/123', 2), ('/users/456', 1), ('/slow', 1)]:
    sample = f'flask_http_request_duration_seconds_count{{method="GET",path="{path}",status="200"}} {float(count)}'
    assert sample in metrics_data, sample
    print(f"  {sample}")

# The /slow handler sleeps 0.5 s, so its recorded duration is at least that long
slow_sum = [line for line in metric_lines
            if line.startswith('flask_http_request_duration_seconds_sum{') and 'path="/slow"' in line]
assert len(slow_sum) == 1 and float(slow_sum[0].rsplit(' ', 1)[1]) >= 0.5, slow_sum

# Shutdown server
server.shutdown()
server_thread.join(timeout=1)
print("Basic Flask metrics integration completed")

In [ ]:
# Example 2: Custom metrics and decorators
from prometheus_flask_exporter import PrometheusMetrics
from flask import Flask, jsonify, request
import time
import random
import threading
from werkzeug.serving import make_server
import requests
from prometheus_client import CollectorRegistry

app2 = Flask(__name__)
# A fresh registry, so this app's metrics do not collide with Example 1's on the default one
metrics2 = PrometheusMetrics(app2, registry=CollectorRegistry())

# Use decorator for specific endpoint monitoring
@app2.route('/api/data')
@metrics2.counter('api_data_requests_v2', 'API data requests', labels={'version': 'v1'})
def api_data():
    # Simulate work
    time.sleep(random.uniform(0.1, 0.3))
    return jsonify({'data': [1, 2, 3, 4, 5]})

@app2.route('/api/users')
@metrics2.histogram('user_request_duration_v2', 'User request duration', labels={'operation': 'list'})
def list_users():
    # Simulate varying load
    time.sleep(random.uniform(0.05, 0.2))
    return jsonify({'users': ['alice', 'bob', 'charlie']})

@app2.route('/api/process')
@metrics2.gauge('processing_queue_size_v2', 'Processing queue size')
def process_data():
    # Simulate processing queue
    queue_size = random.randint(0, 50)
    time.sleep(0.1)
    return jsonify({'processed': True, 'queue_size': queue_size})

@app2.route('/api/summary')
@metrics2.summary('request_latency_v2', 'Request latency summary')
def summary_endpoint():
    time.sleep(random.uniform(0.05, 0.15))
    return jsonify({'summary': 'data processed'})

print("Custom metrics and decorators demonstration")
print("=" * 45)

server2 = make_server('127.0.0.1', 0, app2, threaded=True)
_port = server2.socket.getsockname()[1]
server_thread2 = threading.Thread(target=server2.serve_forever)
server_thread2.daemon = True
server_thread2.start()

time.sleep(0.5)

# Generate test traffic
endpoints = [
    f'http://127.0.0.1:{_port}/api/data',
    f'http://127.0.0.1:{_port}/api/users', 
    f'http://127.0.0.1:{_port}/api/process',
    f'http://127.0.0.1:{_port}/api/summary'
]

print("Generating requests to create custom metrics:")
for _ in range(3):
    for endpoint in endpoints:
        response = requests.get(endpoint, timeout=2)
        endpoint_name = endpoint.split('/')[-1]
        print(f"  {endpoint_name} -> {response.status_code}")
        assert response.status_code == 200, (endpoint, response.status_code)

# Check custom metrics
metrics_response = requests.get(f'http://127.0.0.1:{_port}/metrics', timeout=2)
assert metrics_response.status_code == 200, metrics_response.status_code
metrics_text = metrics_response.text

# Look for decorator-created metrics
decorator_metrics = [
    'api_data_requests_v2',
    'user_request_duration_v2', 
    'processing_queue_size_v2',
    'request_latency_v2'
]

print(f"\nDecorator-based custom metrics found:")
for metric_name in decorator_metrics:
    # Extract metric lines
    lines = [line for line in metrics_text.split('\n') 
            if metric_name in line and not line.startswith('#')]
    assert lines, f"{metric_name} missing from /metrics"
    print(f"  ✓ {metric_name}: {len(lines)} data points")
    print(f"    Example: {lines[0][:80]}...")

# Check for different metric types (each endpoint was hit 3 times above)
print(f"\nMetric types detected:")
assert 'api_data_requests_v2_total{version="v1"} 3.0' in metrics_text
print(f"  ✓ Counter metrics (api_data_requests_v2)")
assert 'user_request_duration_v2_bucket' in metrics_text
assert 'user_request_duration_v2_count{operation="list"} 3.0' in metrics_text
print(f"  ✓ Histogram metrics (user_request_duration_v2)")
assert 'processing_queue_size_v2 0.0' in metrics_text  # in-progress gauge: back to 0 once each request ends
print(f"  ✓ Gauge metrics (processing_queue_size_v2)")
assert 'request_latency_v2_count 3.0' in metrics_text and 'request_latency_v2_sum' in metrics_text
print(f"  ✓ Summary metrics (request_latency_v2)")

server2.shutdown()
server_thread2.join(timeout=1)
print("✓ Custom metrics and decorators demonstration completed!")

In [ ]:
# Example 3: Multi-endpoint monitoring and error tracking
from prometheus_flask_exporter import PrometheusMetrics
from flask import Flask, jsonify, request, abort
import time
import random
import threading
from werkzeug.serving import make_server
import requests
from prometheus_client import CollectorRegistry

app4 = Flask(__name__)
# A FRESH registry: on the shared default one, the exporter finds an earlier example's defaults already registered
# and silently skips exporting its own (`except ValueError: return`), so this app's requests would not be counted.
metrics4 = PrometheusMetrics(app4, registry=CollectorRegistry())

# Track exceptions automatically
metrics4.track_exceptions = True

@app4.route('/api/success')
def success_endpoint():
    time.sleep(random.uniform(0.01, 0.1))
    return jsonify({'status': 'success', 'data': 'Operation completed'})

@app4.route('/api/random-error')
def random_error():
    """Endpoint that randomly succeeds or fails"""
    if random.random() < 0.3:  # 30% chance of error
        abort(500, description="Simulated server error")
    
    return jsonify({'status': 'success', 'random_value': random.randint(1, 100)})

@app4.route('/api/client-error')
def client_error():
    """Endpoint that returns client errors"""
    if 'valid' not in request.args:
        abort(400, description="Missing 'valid' parameter")
    
    return jsonify({'status': 'valid request received'})

@app4.route('/api/not-found-test')
def trigger_404():
    """Endpoint that triggers 404"""
    if random.random() < 0.5:
        abort(404, description="Resource not found")
    
    return jsonify({'found': True})

print("Multi-endpoint monitoring and error tracking")
print("=" * 45)

# Start server
# PORT 0 = kernel-assigned, and `_port` is rebound to THIS server (as in the Example 1 server setup).
server4 = make_server('127.0.0.1', 0, app4, threaded=True)
_port = server4.socket.getsockname()[1]
server_thread4 = threading.Thread(target=server4.serve_forever)
server_thread4.daemon = True
server_thread4.start()

time.sleep(0.5)

# Generate mixed success/error traffic
test_scenarios = [
    (f'http://127.0.0.1:{_port}/api/success', 'Success endpoint'),
    (f'http://127.0.0.1:{_port}/api/random-error', 'Random-failure endpoint'),
    (f'http://127.0.0.1:{_port}/api/client-error', 'Param check (missing)'),
    (f'http://127.0.0.1:{_port}/api/client-error?valid=true', 'Param check (present)'),
    (f'http://127.0.0.1:{_port}/api/not-found-test', 'Not found test'),
    (f'http://127.0.0.1:{_port}/api/nonexistent', 'Nonexistent endpoint')
]

print("Testing endpoints with mixed success/error scenarios:")
results = {'success': 0, 'client_error': 0, 'server_error': 0}

for _ in range(2):  # Run scenarios twice
    for url, description in test_scenarios:
        try:
            response = requests.get(url, timeout=2)
            if response.status_code == 200:
                results['success'] += 1
                print(f"  ✓ {description}: {response.status_code}")
            elif 400 <= response.status_code < 500:
                results['client_error'] += 1
                print(f"  ⚠ {description}: {response.status_code} (client error)")
            else:
                results['server_error'] += 1
                print(f"  ✗ {description}: {response.status_code} (server error)")
                
        except requests.exceptions.RequestException as e:
            results['server_error'] += 1
            print(f"  ✗ {description}: Connection error")

print(f"\nTraffic summary: {results['success']} success, {results['client_error']} client errors, {results['server_error']} server errors")

# Analyze error metrics
metrics_response = requests.get(f'http://127.0.0.1:{_port}/metrics', timeout=2)
assert metrics_response.status_code == 200, metrics_response.status_code
metrics_text = metrics_response.text

print("\nError tracking analysis:")

# Look for automatic error/exception metrics
error_metrics = [
    'flask_http_request_total',
    'flask_http_request_exceptions_total',
    'flask_http_request_duration_seconds'
]

for metric_name in error_metrics:
    assert metric_name in metrics_text, f"{metric_name} missing from /metrics"
    lines = [line for line in metrics_text.split('\n') 
            if metric_name in line and not line.startswith('#')]
    if lines:
        print(f"  ✓ {metric_name}: {len(lines)} data points")
        # Show first example
        print(f"    Example: {lines[0][:100]}...")

# Count HTTP status codes in metrics
status_codes = ['200', '400', '404', '500']
print(f"\nHTTP status code tracking:")
for code in status_codes:
    code_count = metrics_text.count(f'status="{code}"')
    if code_count > 0:
        print(f"  Status {code}: {code_count} occurrences in metrics")
# Deterministic outcomes only (random-error and not-found-test are random): 2 x /api/success -> 200,
# 2 x client-error without params -> 400, 2 x /api/nonexistent -> 404.
assert 'flask_http_request_duration_seconds_count{method="GET",path="/api/success",status="200"} 2.0' in metrics_text
assert 'flask_http_request_duration_seconds_count{method="GET",path="/api/client-error",status="400"} 2.0' in metrics_text
assert 'status="404"' in metrics_text
assert sum(results.values()) == 2 * len(test_scenarios)

# Check for exception tracking
if 'flask_http_request_exceptions_total' in metrics_text:
    exc_lines = [line for line in metrics_text.split('\n') 
                if 'flask_http_request_exceptions_total' in line and not line.startswith('#')]
    if exc_lines:
        print(f"\nException tracking enabled:")
        for line in exc_lines[:3]:
            print(f"  {line[:100]}...")

server4.shutdown()
server_thread4.join(timeout=1)
print("✓ Multi-endpoint monitoring and error tracking completed!")

In [ ]:
# Example 4: Production-ready monitoring with performance analysis
from prometheus_flask_exporter import PrometheusMetrics
from flask import Flask, jsonify, request, g
import time
import random
import threading
from werkzeug.serving import make_server
from functools import wraps
import psutil
import os
import requests
from prometheus_client import CollectorRegistry

app5 = Flask(__name__)

# Production-ready configuration with custom buckets and static labels
metrics5 = PrometheusMetrics(
    app5,
    buckets=[0.01, 0.025, 0.05, 0.075, 0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 10.0],
    static_labels={'service': 'api-service', 'environment': 'production'},
    registry=CollectorRegistry(),  # fresh registry, so this app's defaults are exported (see Example 3)
)

def performance_monitor(f):
    """Decorator for detailed performance monitoring"""
    @wraps(f)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        
        # Simulate database operation
        time.sleep(random.uniform(0.01, 0.05))
        
        result = f(*args, **kwargs)
        
        duration = time.time() - start_time
        # Performance data is automatically tracked by PrometheusMetrics
        
        return result
    
    return wrapper

@app5.route('/api/users')
@metrics5.histogram('user_endpoint_duration', 'User endpoint duration', 
                    labels={'operation': lambda: 'list_users'})
@performance_monitor
def get_users():
    users = [{'id': i, 'name': f'User{i}'} for i in range(1, 11)]
    return jsonify({'users': users, 'cached': random.random() > 0.3})

@app5.route('/api/orders')
@metrics5.histogram('order_endpoint_duration', 'Order endpoint duration',
                    labels={'operation': lambda: 'list_orders'})
@performance_monitor  
def get_orders():
    # Simulate varying database load
    order_count = random.randint(10, 100)
    orders = [{'id': i, 'amount': random.randint(10, 500)} for i in range(order_count)]
    return jsonify({'orders': orders[:20], 'total_count': order_count})

@app5.route('/api/analytics')
@metrics5.histogram('analytics_endpoint_duration', 'Analytics endpoint duration',
                    labels={'operation': lambda: 'aggregate'})
@performance_monitor
def get_analytics():
    # Simulate expensive analytics query
    time.sleep(random.uniform(0.1, 0.3))
    
    return jsonify({
        'metrics': {
            'total_users': random.randint(1000, 5000),
            'total_orders': random.randint(100, 1000),
            'revenue': random.randint(10000, 100000)
        }
    })

@app5.route('/api/health')
@metrics5.do_not_track()
def health_check():
    """Health check endpoint - not tracked"""
    return jsonify({
        'status': 'healthy',
        'cpu': psutil.cpu_percent(),
        'memory': psutil.virtual_memory().percent
    })

print("Production-ready monitoring with performance analysis")
print("=" * 52)

# Start server
# PORT 0 = kernel-assigned, and `_port` is rebound to THIS server (as in the Example 1 server setup).
server5 = make_server('127.0.0.1', 0, app5, threaded=True)
_port = server5.socket.getsockname()[1]
server_thread5 = threading.Thread(target=server5.serve_forever)
server_thread5.daemon = True
server_thread5.start()

time.sleep(0.5)

# Simulate production traffic patterns
endpoints = [
    f'http://127.0.0.1:{_port}/api/users',      # High frequency
    f'http://127.0.0.1:{_port}/api/orders',     # Medium frequency
    f'http://127.0.0.1:{_port}/api/analytics',  # Low frequency, expensive
    f'http://127.0.0.1:{_port}/api/health'      # Not tracked
]

print("Simulating production traffic patterns:")

# Simulate realistic traffic distribution
traffic_patterns = [
    (endpoints[0], 8),   # Users endpoint - 80% of traffic
    (endpoints[1], 3),   # Orders endpoint - 15% of traffic  
    (endpoints[2], 1),   # Analytics endpoint - 5% of traffic
    (endpoints[3], 2)    # Health checks - not tracked
]

total_requests = 0
for endpoint, frequency in traffic_patterns:
    for _ in range(frequency):
        response = requests.get(endpoint, timeout=3)
        assert response.status_code == 200, (endpoint, response.status_code)
        total_requests += 1
        endpoint_name = endpoint.split('/')[-1]
        print(f"  {endpoint_name}: {response.status_code} ({len(response.content)} bytes)")

print(f"Total requests sent: {total_requests}")

# Analyze production metrics
metrics_response = requests.get(f'http://127.0.0.1:{_port}/metrics', timeout=2)
assert metrics_response.status_code == 200, metrics_response.status_code
metrics_text = metrics_response.text

print("\nProduction metrics analysis:")

# Custom endpoint metrics, with the request count each one must have seen
custom_metrics = {
    'user_endpoint_duration': 8,
    'order_endpoint_duration': 3,
    'analytics_endpoint_duration': 1,
}

print("Custom endpoint metrics:")
for metric, expected_count in custom_metrics.items():
    assert metric in metrics_text, f"{metric} missing from /metrics"
    count = len([line for line in metrics_text.split('\n') 
                if metric in line and not line.startswith('#')])
    print(f"  ✓ {metric}: {count} data points")
    count_line = [line for line in metrics_text.split('\n') if line.startswith(f'{metric}_count{{')]
    assert len(count_line) == 1 and count_line[0].endswith(f' {float(expected_count)}'), count_line

# Check for standard Flask metrics
standard_metrics = [
    'flask_http_request_total',
    'flask_http_request_duration_seconds'
]

print("\nStandard Flask metrics:")
for metric in standard_metrics:
    assert metric in metrics_text, f"{metric} missing from /metrics"
    print(f"  ✓ {metric}: Found")

# Performance histogram buckets
histogram_buckets = metrics_text.count('_bucket{')
print(f"\nHistogram buckets tracked: {histogram_buckets}")
assert histogram_buckets > 0

# Static labels
assert 'service="api-service"' in metrics_text
print("✓ Static service labels applied")

assert 'environment="production"' in metrics_text
print("✓ Environment labels applied")

# Check custom buckets
custom_bucket_values = ['0.01', '0.025', '0.075', '2.5']
bucket_found = sum(1 for val in custom_bucket_values if f'le="{val}"' in metrics_text)
print(f"Custom histogram buckets: {bucket_found}/{len(custom_bucket_values)} found")
assert bucket_found == len(custom_bucket_values)

# Verify health endpoint is not tracked (it WAS requested twice above)
assert '/api/health' not in metrics_text and 'health_check' not in metrics_text
print("✓ Health endpoint correctly excluded from metrics")

server5.shutdown()
server_thread5.join(timeout=1)
print("✓ Production-ready monitoring completed!")